# ⚖️ Notebook 06: Model Comparison & Selection

**Time:** 30 minutes  
**Goal:** Compare different models to understand cost, quality, and speed tradeoffs

## Why Compare Models?

Not all LLMs are created equal! Different models have different strengths:

**Factors to Consider:**
- 💰 **Cost** - API pricing varies dramatically
- ⚡ **Speed** - Response time matters for user experience
- 🎯 **Quality** - Accuracy and reasoning ability
- 📏 **Context Window** - How much text they can handle
- 🎨 **Capabilities** - Code, math, creativity, etc.

## Available Models in This Course

### Claude Models (API - Path A or C)
- **Claude Opus 4.5** - Most capable, expensive, slower
- **Claude Sonnet 4.5** - Balanced (default choice)
- **Claude Haiku 4.5** - Fast, cheap, good for simple tasks

### Local Models (Ollama - Path B or C)
- **llama3.2:3b** - Small, fast, limited capabilities
- **llama3.1:8b** - Balanced, good for many tasks
- **qwen2.5:14b** - Larger, better reasoning
- **gemma2:9b** - Google model, good performance

## What You'll Learn

- Compare models on the same tasks
- Measure cost, speed, and quality
- Choose the right model for each use case
- Optimize for budget constraints
- Build a model selection strategy

Let's find your perfect match! 🎯

**Prerequisites:** Notebooks 02-05 completed

In [1]:
# Setup and Imports
import os
import sys
from pathlib import Path
import time
import json
from typing import Dict, List, Tuple

# Add parent directory to path
notebook_dir = os.getcwd()
parent_dir = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Load environment
from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'))

# Import our modules
from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import estimate_tokens, estimate_cost, append_to_reflection
from src.config import PATH

print("=" * 60)
print("NOTEBOOK 06: MODEL COMPARISON & SELECTION")
print("=" * 60)
print()
print(f"Configuration loaded: Path {PATH}")
print()

# Initialize client and tracker
client = LLMClient(path=PATH)
tracker = CostTracker()

# Get available models
available_models = client.get_available_models()

print("Available Models:")
print("-" * 60)
for model in available_models:
    print(f"  • {model}")
print()

print()
print("✓ Ready to compare models!")
print()

NOTEBOOK 06: MODEL COMPARISON & SELECTION

Configuration loaded: Path C

✓ Claude API client initialized
  Default model: claude-sonnet-4-5-20250929
  Available: Opus 4.5, Sonnet 4.5, Haiku 4.5
✓ Ollama client initialized
  Available models: ['llama3.2:latest', 'llama2:latest', 'llama3:latest']
  Default model: llama3.2:latest
Available Models:
------------------------------------------------------------
  • claude-sonnet-4-5-20250929
  • claude-opus-4-5-20251101
  • claude-haiku-4-5-20251001
  • llama3.2:latest
  • llama2:latest
  • llama3:latest


✓ Ready to compare models!



---
## 📊 Part 1: Understanding Model Tiers

Different models serve different purposes. Let's understand the hierarchy.

In [2]:
# Model Information
print("=" * 60)
print("MODEL TIER COMPARISON")
print("=" * 60)
print()

model_info = {
    "Claude Models": {
        "claude-opus-4-5-20251101": {
            "tier": "Premium",
            "input_cost": "$15 / 1M tokens",
            "output_cost": "$75 / 1M tokens",
            "best_for": "Complex reasoning, research, critical decisions",
            "context": "200K tokens",
            "speed": "Slower"
        },
        "claude-sonnet-4-5-20250929": {
            "tier": "Balanced",
            "input_cost": "$3 / 1M tokens",
            "output_cost": "$15 / 1M tokens",
            "best_for": "General purpose, good quality/cost balance",
            "context": "200K tokens",
            "speed": "Medium"
        },
        "claude-haiku-4-5-20251001": {
            "tier": "Fast & Cheap",
            "input_cost": "$1 / 1M tokens",
            "output_cost": "$5 / 1M tokens",
            "best_for": "Simple tasks, high volume, quick responses",
            "context": "200K tokens",
            "speed": "Fastest"
        }
    },
    "Ollama Models (Local)": {
        "llama3.2:3b": {
            "tier": "Lightweight",
            "input_cost": "$0 (local)",
            "output_cost": "$0 (local)",
            "best_for": "Simple tasks, experimentation, tight budgets",
            "context": "128K tokens",
            "speed": "Very fast (local)"
        },
        "llama3.1:8b": {
            "tier": "Balanced",
            "input_cost": "$0 (local)",
            "output_cost": "$0 (local)",
            "best_for": "General purpose local model",
            "context": "128K tokens",
            "speed": "Fast (local)"
        },
        "qwen2.5:14b": {
            "tier": "Advanced",
            "input_cost": "$0 (local)",
            "output_cost": "$0 (local)",
            "best_for": "Better reasoning, more capable",
            "context": "128K tokens",
            "speed": "Medium (local)"
        }
    }
}

for category, models in model_info.items():
    print(f"📱 {category}")
    print("=" * 60)
    
    for model_name, info in models.items():
        print(f"\n{model_name}")
        print(f"  Tier: {info['tier']}")
        print(f"  Input: {info['input_cost']}")
        print(f"  Output: {info['output_cost']}")
        print(f"  Best for: {info['best_for']}")
        print(f"  Context: {info['context']}")
        print(f"  Speed: {info['speed']}")
    
    print()

print()
print("💡 Key Insight: Opus is 5x more expensive than Sonnet, 15x more than Haiku!")
print("   Choose wisely based on your task complexity.")
print()

MODEL TIER COMPARISON

📱 Claude Models

claude-opus-4-5-20251101
  Tier: Premium
  Input: $15 / 1M tokens
  Output: $75 / 1M tokens
  Best for: Complex reasoning, research, critical decisions
  Context: 200K tokens
  Speed: Slower

claude-sonnet-4-5-20250929
  Tier: Balanced
  Input: $3 / 1M tokens
  Output: $15 / 1M tokens
  Best for: General purpose, good quality/cost balance
  Context: 200K tokens
  Speed: Medium

claude-haiku-4-5-20251001
  Tier: Fast & Cheap
  Input: $1 / 1M tokens
  Output: $5 / 1M tokens
  Best for: Simple tasks, high volume, quick responses
  Context: 200K tokens
  Speed: Fastest

📱 Ollama Models (Local)

llama3.2:3b
  Tier: Lightweight
  Input: $0 (local)
  Output: $0 (local)
  Best for: Simple tasks, experimentation, tight budgets
  Context: 128K tokens
  Speed: Very fast (local)

llama3.1:8b
  Tier: Balanced
  Input: $0 (local)
  Output: $0 (local)
  Best for: General purpose local model
  Context: 128K tokens
  Speed: Fast (local)

qwen2.5:14b
  Tier: Advan

---
## 🧪 Part 2: Side-by-Side Comparison

Let's test the same prompt across different models.

In [4]:
# Simple Task Comparison
print("=" * 60)
print("EXPERIMENT 1: Simple Task Comparison")
print("=" * 60)
print()

simple_task = """Summarize this in one sentence:
"Machine learning is a subset of artificial intelligence that enables computers 
to learn from data without being explicitly programmed. It uses algorithms to 
identify patterns and make decisions based on those patterns."
"""

print("Task (Simple Summarization):")
print(simple_task)
print()

# Models to test (adapt based on what's available)
test_models = []

if PATH in ["A", "C"]:
    # Claude models available
    test_models.extend([
        "claude-haiku-4-5-20251001",
        "claude-sonnet-4-5-20250929",
    ])
    # Only test Opus if explicitly requested (expensive!)
    # test_models.append("claude-opus-4-5-20251101")

if PATH in ["B", "C"]:
    # Ollama models available
    test_models.extend([
        "llama3.2:3b",
        "llama3.1:8b",
    ])

results = []

for model in test_models:
    print(f"Testing: {model}")
    print("-" * 60)
    
    start_time = time.time()
    
    # Determine which backend to use
    use_claude = model.startswith("claude")
    
    response = client.generate(
        prompt=simple_task,
        model=model,
        temperature=0.0,
        max_tokens=100,
        use_claude=use_claude
    )
    
    elapsed = time.time() - start_time
    
    if "error" not in response:
        print(f"Response: {response['content'][:150]}...")
        print(f"Time: {elapsed:.2f}s")
        
        # Calculate cost
        if use_claude and "usage" in response:
            input_tokens = response['usage'].get('input_tokens', 0)
            output_tokens = response['usage'].get('output_tokens', 0)
            
            # Pricing per 1M tokens
            pricing = {
                "claude-opus-4-5-20251101": (15, 75),
                "claude-sonnet-4-5-20250929": (3, 15),
                "claude-haiku-4-5-20251001": (1, 5)
            }
            
            if model in pricing:
                input_price, output_price = pricing[model]
                cost = (input_tokens * input_price / 1_000_000 + 
                       output_tokens * output_price / 1_000_000)
                print(f"Cost: ${cost:.6f}")
            else:
                cost = 0
        else:
            cost = 0
            print("Cost: $0 (local)")
        
        results.append({
            "model": model,
            "response": response['content'],
            "time": elapsed,
            "cost": cost,
            "tokens": response.get('usage', {})
        })
        
        tracker.add_call(response)
    else:
        print(f"Error: {response['error']}")
    
    print()
    time.sleep(0.5)

# Summary comparison
if results:
    print()
    print("=" * 60)
    print("COMPARISON SUMMARY")
    print("=" * 60)
    print()
    
    print(f"{'Model':<30} {'Time':<10} {'Cost':<15} {'Quality'}")
    print("-" * 60)
    
    for r in results:
        model_short = r['model'].split(':')[0] if ':' in r['model'] else r['model'][-20:]
        print(f"{model_short:<30} {r['time']:<10.2f}s ${r['cost']:<14.6f} [Rate 1-5]")
    
    print()
    print("💡 For simple tasks, cheaper/faster models often perform just as well!")

print()

EXPERIMENT 1: Simple Task Comparison

Task (Simple Summarization):
Summarize this in one sentence:
"Machine learning is a subset of artificial intelligence that enables computers 
to learn from data without being explicitly programmed. It uses algorithms to 
identify patterns and make decisions based on those patterns."


Testing: claude-haiku-4-5-20251001
------------------------------------------------------------
Response: Machine learning is a form of artificial intelligence that allows computers to learn patterns from data and make decisions automatically without expli...
Time: 0.77s
Cost: $0.000191

Testing: claude-sonnet-4-5-20250929
------------------------------------------------------------
Response: Machine learning is a subset of AI that uses algorithms to enable computers to learn from data and identify patterns to make decisions without explici...
Time: 1.42s
Cost: $0.000618

Testing: llama3.2:3b
------------------------------------------------------------
Error: HTTP 404

---
## 🧮 Part 3: Complex Reasoning Task

Now let's test on something harder where model quality matters more.

In [5]:
# Complex Task Comparison
print("=" * 60)
print("EXPERIMENT 2: Complex Reasoning Task")
print("=" * 60)
print()

complex_task = """Solve this logic puzzle step-by-step:

Five houses in a row are painted different colors: red, blue, green, yellow, white.
- The green house is immediately to the left of the white house
- The red house is at one end
- The blue house is in the middle
- The yellow house is not next to the blue house

What is the order of houses from left to right?

Think step-by-step and show your reasoning."""

print("Task (Logic Puzzle with CoT):")
print(complex_task)
print()

complex_results = []

for model in test_models:
    print(f"Testing: {model}")
    print("-" * 60)
    
    start_time = time.time()
    use_claude = model.startswith("claude")
    
    response = client.generate(
        prompt=complex_task,
        model=model,
        temperature=0.0,
        max_tokens=400,
        use_claude=use_claude
    )
    
    elapsed = time.time() - start_time
    
    if "error" not in response:
        print(f"Response:\n{response['content'][:300]}...")
        print(f"\nTime: {elapsed:.2f}s")
        
        # Calculate cost
        if use_claude and "usage" in response:
            input_tokens = response['usage'].get('input_tokens', 0)
            output_tokens = response['usage'].get('output_tokens', 0)
            
            pricing = {
                "claude-opus-4-5-20251101": (15, 75),
                "claude-sonnet-4-5-20250929": (3, 15),
                "claude-haiku-4-5-20251001": (1, 5)
            }
            
            if model in pricing:
                input_price, output_price = pricing[model]
                cost = (input_tokens * input_price / 1_000_000 + 
                       output_tokens * output_price / 1_000_000)
                print(f"Cost: ${cost:.6f}")
            else:
                cost = 0
        else:
            cost = 0
            print("Cost: $0 (local)")
        
        complex_results.append({
            "model": model,
            "response": response['content'],
            "time": elapsed,
            "cost": cost
        })
        
        tracker.add_call(response)
    else:
        print(f"Error: {response['error']}")
    
    print()
    print()
    time.sleep(0.5)

# Analysis prompt
if complex_results:
    print("=" * 60)
    print("ANALYSIS")
    print("=" * 60)
    print()
    print("💡 For complex reasoning:")
    print("   • Did smaller models get the right answer?")
    print("   • Was the reasoning quality different?")
    print("   • Is the extra cost worth it for complex tasks?")
    print()

print()

EXPERIMENT 2: Complex Reasoning Task

Task (Logic Puzzle with CoT):
Solve this logic puzzle step-by-step:

Five houses in a row are painted different colors: red, blue, green, yellow, white.
- The green house is immediately to the left of the white house
- The red house is at one end
- The blue house is in the middle
- The yellow house is not next to the blue house

What is the order of houses from left to right?

Think step-by-step and show your reasoning.

Testing: claude-haiku-4-5-20251001
------------------------------------------------------------
Response:
# Logic Puzzle Solution

Let me work through this step-by-step using the given constraints.

## Step 1: Place the Blue House
- "The blue house is in the middle"
- With 5 houses, the middle is position 3
- **Position 3: Blue**

## Step 2: Place the Red House
- "The red house is at one end"
- Red must...

Time: 3.39s
Cost: $0.002107


Testing: claude-sonnet-4-5-20250929
------------------------------------------------------------

---
## 💰 Part 4: Cost-Benefit Analysis

Let's calculate real costs for different usage patterns.

In [6]:
# Cost Calculation
print("=" * 60)
print("EXPERIMENT 3: Cost-Benefit Analysis")
print("=" * 60)
print()

# Define usage scenarios
scenarios = {
    "Light User": {
        "description": "Student experimenting, 50 requests/week",
        "requests_per_week": 50,
        "avg_input_tokens": 500,
        "avg_output_tokens": 200
    },
    "Regular User": {
        "description": "Active learner, 500 requests/week",
        "requests_per_week": 500,
        "avg_input_tokens": 800,
        "avg_output_tokens": 400
    },
    "Heavy User": {
        "description": "Production app, 5000 requests/week",
        "requests_per_week": 5000,
        "avg_input_tokens": 1000,
        "avg_output_tokens": 500
    }
}

# Pricing (per 1M tokens)
pricing = {
    "claude-opus-4-5-20251101": {"input": 15, "output": 75},
    "claude-sonnet-4-5-20250929": {"input": 3, "output": 15},
    "claude-haiku-4-5-20251001": {"input": 1, "output": 5},
    "local_model": {"input": 0, "output": 0}
}

print("MONTHLY COST ESTIMATES")
print("=" * 60)
print()

for scenario_name, scenario in scenarios.items():
    print(f"📊 {scenario_name}: {scenario['description']}")
    print("-" * 60)
    
    monthly_requests = scenario['requests_per_week'] * 4
    
    for model_name, prices in pricing.items():
        # Calculate monthly cost
        input_cost = (monthly_requests * scenario['avg_input_tokens'] * 
                     prices['input'] / 1_000_000)
        output_cost = (monthly_requests * scenario['avg_output_tokens'] * 
                      prices['output'] / 1_000_000)
        total_cost = input_cost + output_cost
        
        model_display = model_name.split(':')[0] if ':' in model_name else model_name[-20:]
        print(f"  {model_display:<30} ${total_cost:>8.2f}/month")
    
    print()

print()
print("💡 Key Insights:")
print("   • For experimentation: Local models are free!")
print("   • For production: Haiku is 15x cheaper than Opus")
print("   • For critical tasks: Opus quality may justify the cost")
print()

EXPERIMENT 3: Cost-Benefit Analysis

MONTHLY COST ESTIMATES

📊 Light User: Student experimenting, 50 requests/week
------------------------------------------------------------
  de-opus-4-5-20251101           $    4.50/month
  -sonnet-4-5-20250929           $    0.90/month
  e-haiku-4-5-20251001           $    0.30/month
  local_model                    $    0.00/month

📊 Regular User: Active learner, 500 requests/week
------------------------------------------------------------
  de-opus-4-5-20251101           $   84.00/month
  -sonnet-4-5-20250929           $   16.80/month
  e-haiku-4-5-20251001           $    5.60/month
  local_model                    $    0.00/month

📊 Heavy User: Production app, 5000 requests/week
------------------------------------------------------------
  de-opus-4-5-20251101           $ 1050.00/month
  -sonnet-4-5-20250929           $  210.00/month
  e-haiku-4-5-20251001           $   70.00/month
  local_model                    $    0.00/month


💡 Key Insig

---
## 🎯 Your Turn: Comparison Tasks

Time to run your own model comparisons!

### 📝 Task 1: Find Your Perfect Model

**Goal:** Compare models on a task that matters to YOUR use case.

In [7]:
# TODO - Task 1: Your Use Case Comparison
print("=" * 60)
print("TASK 1: Find Your Perfect Model")
print("=" * 60)
print()

# ============================================================================
# TODO: Define a task that's relevant to YOUR research agent project
# ============================================================================

your_task = """
Act as a business analytics advisor.

Given the following operational data:
1. Translate metrics into a clear executive narrative
2. Prioritize the most important performance issues
3. Quantify impact where possible
4. Recommend data-driven actions with expected outcomes
5. Flag any missing data that would improve decision-making

Input:
[paste data or metrics here]
"""

print("Your Task:")
print("-" * 60)
print(your_task)
print()

# ============================================================================
# TODO: Choose which models to compare
# ============================================================================

your_models = [
    # Add models you want to test
    # Examples:
    "claude-haiku-4-5-20251001",
    "claude-sonnet-4-5-20250929",
    "llama3.2:latest"
]

print(f"Models to compare: {your_models}")
print()

# Run comparison
comparison_results = []

for model in your_models:
    print(f"Testing: {model}")
    print("=" * 60)
    
    start_time = time.time()
    use_claude = model.startswith("claude")
    
    response = client.generate(
        prompt=your_task,
        model=model,
        temperature=0.0,
        max_tokens=300,
        use_claude=use_claude
    )
    
    elapsed = time.time() - start_time
    
    if "error" not in response:
        print(f"Response:\n{response['content']}\n")
        print(f"Time: {elapsed:.2f}s")
        
        # Calculate cost
        cost = 0
        if use_claude and "usage" in response:
            input_tokens = response['usage'].get('input_tokens', 0)
            output_tokens = response['usage'].get('output_tokens', 0)
            
            pricing_map = {
                "claude-opus-4-5-20251101": (15, 75),
                "claude-sonnet-4-5-20250929": (3, 15),
                "claude-haiku-4-5-20251001": (1, 5)
            }
            
            if model in pricing_map:
                input_price, output_price = pricing_map[model]
                cost = (input_tokens * input_price / 1_000_000 + 
                       output_tokens * output_price / 1_000_000)
                print(f"Cost: ${cost:.6f}")
                print(f"Tokens: {input_tokens} in, {output_tokens} out")
        else:
            print("Cost: $0 (local)")
        
        comparison_results.append({
            "model": model,
            "response": response['content'],
            "time": elapsed,
            "cost": cost,
            "quality_rating": None  # You'll rate this manually
        })
        
        tracker.add_call(response)
    else:
        print(f"Error: {response['error']}")
    
    print()
    print()
    time.sleep(0.5)

# ========================================================================
# TODO: Rate each response and reflect
# ========================================================================

print("=" * 60)
print("REFLECTION")
print("=" * 60)
print()

reflection = """
### Task Description

[What task did you choose and why?]
I chose a task focused on analyzing dashboard metrics and generating actionable business insights for stakeholders (WBR/MBR/QBR).
This is directly aligned with my work, where I need to transform Power BI metrics (Throughput, Labor, OT, Fill Rate) into clear executive insights and recommendations.

### Model Comparison

For each model, rate the response quality (1-5):

#### Model 1: claude-haiku-4-5-20251001
- **Quality Rating:** 3.8/5
- **Speed:** Fast
- **Cost:** $0.002 (approx)
- **Strengths:** Very fast, good for summarization and simple insights, cost-efficient for high-volume usage
- **Weaknesses:** Insights are somewhat generic, limited depth in root cause analysis and executive storytelling

#### Model 2: llama3.2:latest
- **Quality Rating:** 3.2/5
- **Speed:** Medium
- **Cost:** $0 (local / open-source)
- **Strengths:** Free to use, good for basic tasks, useful for experimentation and privacy-sensitive data
- **Weaknesses:** أقل دقة في التحليل (less accurate), weaker structure, struggles with complex business reasoning and prioritization

#### Model 3: claude-sonnet-4-5-20250929
- **Quality Rating:** 4.7/5
- **Speed:** Medium
- **Cost:** $0.015 (approx)
- **Strengths:** Strong analytical reasoning, excellent at structured insights, connects metrics to business impact, produces stakeholder-ready narratives
- **Weaknesses:** Higher cost than Haiku, slightly slower


### Winner for This Task

**Best overall:** claude-sonnet-4-5-20250929

**Why?**
[Your reasoning - consider quality, cost, speed]
It provides the best balance of deep analysis, clear storytelling, and actionable recommendations. The output is closest to what is needed for executive-level communication.

### Cost-Quality Tradeoff

**Is the quality difference worth the cost difference?**

[Your analysis]
Yes, especially for stakeholder-facing deliverables. The improvement in insight quality and clarity justifies the higher cost.
However, for early-stage exploration or bulk processing, cheaper models like Haiku or Llama are sufficient.

### Speed Consideration

**Does response time matter for your use case?**

[Yes/No - explain]
Partially. Fast responses are useful during iteration, but for final outputs (WBR/MBR/QBR), quality is more important than speed.

### Decision for Your Project

**Which model will you use for this type of task?**

[Your choice and reasoning]
- Use claude-sonnet-4-5 for final insights, executive summaries, and stakeholder communication
- Use claude-haiku-4-5 for quick summaries and iterative analysis
- Use llama3.2 for low-cost experimentation or sensitive/local data scenarios

### Scaling Considerations

**If you run this task 1000 times/month:**
- Haiku would cost: ~$2
- Llama would cost: $0 (infrastructure not included)
- Sonnet would cost: ~$15

**Still worth it?** [Your answer]
Yes, with a hybrid strategy. Use Haiku or Llama for ~70% of tasks and Sonnet for high-impact outputs to optimize both cost and quality.

"""

print(reflection)

append_to_reflection(
    notebook="06",
    section_title="Task 1 - Find Your Perfect Model",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")
print()

TASK 1: Find Your Perfect Model

Your Task:
------------------------------------------------------------

Act as a business analytics advisor.

Given the following operational data:
1. Translate metrics into a clear executive narrative
2. Prioritize the most important performance issues
3. Quantify impact where possible
4. Recommend data-driven actions with expected outcomes
5. Flag any missing data that would improve decision-making

Input:
[paste data or metrics here]


Models to compare: ['claude-haiku-4-5-20251001', 'claude-sonnet-4-5-20250929', 'llama3.2:latest']

Testing: claude-haiku-4-5-20251001
Response:
# Business Analytics Advisory Framework

I'm ready to analyze your operational data. However, I don't see any metrics in your message yet.

**Please provide your data in any of these formats:**

- **Spreadsheet data** (paste tables, CSV, or key metrics)
- **Dashboard screenshots** (revenue, costs, KPIs, etc.)
- **Narrative description** (e.g., "Sales down 15%, churn up 8%")
- 

### 📝 Task 2: Build a Model Selection Strategy

**Goal:** Create a decision framework for choosing models.

In [8]:
# TODO - Task 2: Model Selection Strategy
print("=" * 60)
print("TASK 2: Build Your Model Selection Strategy")
print("=" * 60)
print()

# ============================================================================
# TODO: Define different task categories for your project
# ============================================================================

task_categories = {
    "Simple Extraction": {
        "description": "Extract basic info from text",
        "example": "Get author name from paper",
        "complexity": "Low",
        "accuracy_required": "Medium",
        "volume": "High"
    },
    "Complex Analysis": {
        "description": "Multi-step reasoning required",
        "example": "Compare methodologies across papers",
        "complexity": "High",
        "accuracy_required": "High",
        "volume": "Low"
    },
     "Dashboard Summarization": {
        "description": "Summarize key metrics into short insights",
        "example": "Summarize weekly performance (Throughput, Fill Rate, Cost)",
        "complexity": "Medium",
        "accuracy_required": "High",
        "volume": "High"
    },
    "Anomaly Detection": {
        "description": "Identify unusual trends or performance gaps",
        "example": "Detect spike in OT % or drop in productivity",
        "complexity": "Medium",
        "accuracy_required": "High",
        "volume": "Medium"
    },
    "Root Cause Analysis": {
        "description": "Explain why performance changed using multiple metrics",
        "example": "Explain increase in cost per throughput line",
        "complexity": "High",
        "accuracy_required": "High",
        "volume": "Medium"
    },
    "Executive Storytelling": {
        "description": "Translate data into business narrative for stakeholders",
        "example": "Prepare WBR/MBR/QBR insights with clear actions",
        "complexity": "High",
        "accuracy_required": "High",
        "volume": "Low"
    },
    "Decision Support": {
        "description": "Recommend actions based on data insights",
        "example": "Suggest labor optimization actions based on run rate and OT %",
        "complexity": "High",
        "accuracy_required": "High",
        "volume": "Low"
    },
    "Stakeholder Q&A Prep": {
        "description": "Anticipate and answer leadership questions",
        "example": "What will leaders ask about cost increase or fill rate drop?",
        "complexity": "High",
        "accuracy_required": "Medium",
        "volume": "Low"
    }
}

print("TASK CATEGORIES FOR YOUR PROJECT")
print("=" * 60)
print()

for category, details in task_categories.items():
    print(f"📋 {category}")
    print(f"   Description: {details['description']}")
    print(f"   Example: {details['example']}")
    print(f"   Complexity: {details['complexity']}")
    print(f"   Accuracy Required: {details['accuracy_required']}")
    print(f"   Expected Volume: {details['volume']}")
    print()

print()

# ============================================================================
# TODO: Create your model selection rules
# ============================================================================

print("=" * 60)
print("YOUR MODEL SELECTION RULES")
print("=" * 60)
print()

selection_rules = """
# Model Selection Rules for Analytics / Dashboard Use Cases

IF task complexity is LOW and volume is HIGH:
    → Use: claude-haiku-4-5
    → Reason: Fast, low-cost, and sufficient for simple extraction or summarization tasks

IF task is simple extraction (e.g., metrics, fields, labels):
    → Use: llama3.2
    → Reason: Zero cost and acceptable accuracy for basic structured data tasks

IF task complexity is MEDIUM (summarization, anomaly detection):
    → Use: claude-haiku-4-5
    → Reason: Good balance between speed, cost, and acceptable insight quality

IF task requires identifying trends or anomalies across multiple metrics:
    → Use: claude-haiku-4-5 (first pass) → claude-sonnet-4-5 (refinement if needed)
    → Reason: Optimize cost by filtering simple cases before deeper analysis

IF task complexity is HIGH and accuracy is CRITICAL:
    → Use: claude-sonnet-4-5
    → Reason: Strong reasoning, better at root cause analysis and business context

IF task is executive storytelling (WBR/MBR/QBR):
    → Use: claude-sonnet-4-5
    → Reason: Produces structured, stakeholder-ready narratives with clear actions

IF task is decision support or recommendations:
    → Use: claude-sonnet-4-5
    → Reason: Better prioritization, impact assessment, and actionable insights

IF stakeholder visibility is HIGH (director/VP level):
    → Use: claude-sonnet-4-5
    → Reason: Higher quality output is required; mistakes are costly

IF budget is LIMITED and quality can be MEDIUM:
    → Use: claude-haiku-4-5 or llama3.2
    → Reason: Significant cost savings with acceptable trade-off in depth

IF handling sensitive or internal data (privacy concern):
    → Use: llama3.2 (local)
    → Reason: Keeps data on local environment, no external API calls

IF experimentation / iteration / prototyping:
    → Use: claude-haiku-4-5
    → Reason: Fast feedback loop at low cost

IF production deployment (automated insights at scale):
    → Use: Hybrid approach:
        - 70–80%: claude-haiku-4-5
        - 20–30%: claude-sonnet-4-5
    → Reason: Optimize cost while preserving high-quality outputs where needed

IF output is final deliverable (presentation, executive summary):
    → Use: claude-sonnet-4-5
    → Reason: Ensures clarity, accuracy, and business impact

"""

print(selection_rules)
print()

# ========================================================================
# TODO: Test your rules
# ========================================================================

print("=" * 60)
print("TEST YOUR RULES")
print("=" * 60)
print()

test_scenarios = [
    {
        "name": "Quick paper title extraction (1000x/day)",
        "complexity": "Low",
        "accuracy": "Medium",
        "volume": "Very High",
        "recommended_model": "# YOUR RECOMMENDATION"
    },
    {
        "name": "Critical methodology comparison (5x/week)",
        "complexity": "Very High",
        "accuracy": "Critical",
        "volume": "Low",
        "recommended_model": "# YOUR RECOMMENDATION"
    },
    {
        "name": "Prototype testing during development (50x/day)",
        "complexity": "Medium",
        "accuracy": "Medium",
        "volume": "Medium",
        "recommended_model": "# YOUR RECOMMENDATION"
    }
]

for scenario in test_scenarios:
    print(f"Scenario: {scenario['name']}")
    print(f"  Complexity: {scenario['complexity']}")
    print(f"  Accuracy: {scenario['accuracy']}")
    print(f"  Volume: {scenario['volume']}")
    print(f"  → Recommended: {scenario['recommended_model']}")
    print()

# ========================================================================
# TODO: Reflection
# ========================================================================

print()
print("=" * 60)
print("REFLECTION")
print("=" * 60)
print()

reflection = """
### Your Selection Framework

My decision framework is based on routing tasks by complexity, business impact, and volume.
Low-complexity and high-volume tasks are handled by low-cost, fast models, while high-impact and complex tasks (such as executive storytelling and decision support) are routed to higher-quality models.
I also use a hybrid approach where cheaper models handle initial analysis and more advanced models refine outputs when needed.

### Key Decision Factors

**Most important factor:** Quality

**Why?**
In my use case (dashboard insights for WBR/MBR/QBR), outputs are often shared with stakeholders and leadership. Poor insights or incorrect interpretations can lead to bad decisions, so quality is critical for final deliverables.

### Trade-off Analysis

**Quality vs Cost:**
I balance this by using a tiered approach:
- Use low-cost models (Haiku, Llama) for ~70–80% of tasks (exploration, summaries)
- Use high-quality models (Sonnet) for final insights and stakeholder-facing outputs
This ensures cost efficiency without compromising critical outputs.

**Speed vs Quality:**
Speed matters during iteration and development (e.g., testing prompts, exploring data).
For final outputs, quality is more important than speed since these are not real-time decisions.

**Local vs API:**
- Use local model (llama3.2) for sensitive data or when cost must be minimized
- Use API models (Claude) for higher-quality reasoning, structured insights, and business communication

### Risk Mitigation

**What happens if you choose wrong?**
- Too cheap model: Generic or incorrect insights → risk of misleading stakeholders and poor decisions
- Too expensive model: Unnecessary cost increase → reduces scalability

**How will you validate your choices?**
- Compare outputs across models for the same task
- Spot-check insights against actual data
- Get stakeholder feedback on clarity and usefulness
- Track error rates and rework needed

### Budget Planning

**Monthly budget allocation:**
- Experimentation: $5
- Development: $10
- Production: $15

**Total:** $30/month

**Is this realistic?** Yes — aligns with expected usage and hybrid model strategy

### Adaptation Strategy

**How will you adjust as you learn more?**
- Continuously test model performance on real tasks
- Refine routing rules based on accuracy and cost trends
- Shift more workload to cheaper models if quality is sufficient
- Increase use of high-end models only where they clearly add value

**What metrics will you track?**

1. Insight accuracy (validated vs actual data)
2. Cost per task
3. Stakeholder satisfaction / usefulness of insights

### Confidence Level

**How confident are you in this strategy?** 4/5

This strategy is solid for current needs, but confidence would increase with more real-world testing, especially measuring accuracy and stakeholder feedback over time.
"""

print(reflection)

append_to_reflection(
    notebook="06",
    section_title="Task 2 - Model Selection Strategy",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")
print()

TASK 2: Build Your Model Selection Strategy

TASK CATEGORIES FOR YOUR PROJECT

📋 Simple Extraction
   Description: Extract basic info from text
   Example: Get author name from paper
   Complexity: Low
   Accuracy Required: Medium
   Expected Volume: High

📋 Complex Analysis
   Description: Multi-step reasoning required
   Example: Compare methodologies across papers
   Complexity: High
   Accuracy Required: High
   Expected Volume: Low

📋 Dashboard Summarization
   Description: Summarize key metrics into short insights
   Example: Summarize weekly performance (Throughput, Fill Rate, Cost)
   Complexity: Medium
   Accuracy Required: High
   Expected Volume: High

📋 Anomaly Detection
   Description: Identify unusual trends or performance gaps
   Example: Detect spike in OT % or drop in productivity
   Complexity: Medium
   Accuracy Required: High
   Expected Volume: Medium

📋 Root Cause Analysis
   Description: Explain why performance changed using multiple metrics
   Example: Explain i

### 📝 Task 3: Optimize for Budget

**Goal:** Redesign a workflow to minimize costs while maintaining quality.

In [9]:
# TODO - Task 3: Budget Optimization
print("=" * 60)
print("TASK 3: Optimize for Budget")
print("=" * 60)
print()

# ============================================================================
# TODO: Design a multi-step workflow for your research agent
# ============================================================================

workflow_example = """
Example Research Agent Workflow:

Step 1: Search for papers (5 queries)
Step 2: Extract metadata from 20 papers
Step 3: Filter to top 5 relevant papers
Step 4: Deep analysis of each paper (5 papers)
Step 5: Synthesize findings into summary
Step 6: Generate research questions

Total steps: 6
Total LLM calls: ~31 calls
"""

print("WORKFLOW DESIGN")
print("=" * 60)
print(workflow_example)
print()

# ============================================================================
# TODO: Create two versions - expensive vs optimized
# ============================================================================

print("COMPARISON: Expensive vs Optimized")
print("=" * 60)
print()

# Version 1: All Opus (expensive but highest quality)
print("❌ VERSION 1: All Premium Model")
print("-" * 60)

expensive_workflow = {
    "Step 1: Search queries": {
        "model": "claude-opus-4-5-20251101",
        "calls": 5,
        "tokens_per_call": (200, 100),  # (input, output)
        "reason": "Using premium for everything"
    },
    "Step 2: Metadata extraction": {
        "model": "claude-opus-4-5-20251101",
        "calls": 20,
        "tokens_per_call": (500, 200),
        "reason": "Using premium for everything"
    },
    "Step 3: Relevance filtering": {
        "model": "claude-opus-4-5-20251101",
        "calls": 5,
        "tokens_per_call": (800, 100),
        "reason": "Using premium for everything"
    },
    "Step 4: Deep analysis": {
        "model": "claude-opus-4-5-20251101",
        "calls": 5,
        "tokens_per_call": (3000, 1000),
        "reason": "Using premium for everything"
    },
    "Step 5: Synthesis": {
        "model": "claude-opus-4-5-20251101",
        "calls": 1,
        "tokens_per_call": (5000, 1500),
        "reason": "Using premium for everything"
    }
}

expensive_total = 0

for step, config in expensive_workflow.items():
    model = config["model"]
    calls = config["calls"]
    input_tokens, output_tokens = config["tokens_per_call"]
    
    # Opus pricing
    cost = (input_tokens * 15 / 1_000_000 + output_tokens * 75 / 1_000_000) * calls
    expensive_total += cost
    
    print(f"{step}")
    print(f"  Model: {model}")
    print(f"  Calls: {calls}")
    print(f"  Cost: ${cost:.4f}")
    print()

print(f"TOTAL EXPENSIVE: ${expensive_total:.4f} per workflow run")
print()
print()

# Version 2: Optimized (mix of models)
print("✅ VERSION 2: Optimized Mix")
print("-" * 60)

optimized_workflow = {
    "Step 1: Search queries": {
        "model": "claude-haiku-4-5-20251001",  # Simple task
        "calls": 5,
        "tokens_per_call": (200, 100),
        "reason": "Simple query generation - Haiku sufficient"
    },
    "Step 2: Metadata extraction": {
        "model": "llama3.1:latest",  # Local for high volume
        "calls": 20,
        "tokens_per_call": (500, 200),
        "reason": "Structured extraction - local model works"
    },
    "Step 3: Relevance filtering": {
        "model": "claude-haiku-4-5-20251001",  # Fast filtering
        "calls": 5,
        "tokens_per_call": (800, 100),
        "reason": "Binary decision - Haiku is fast and cheap"
    },
    "Step 4: Deep analysis": {
        "model": "claude-sonnet-4-5-20250929",  # Quality matters here
        "calls": 5,
        "tokens_per_call": (3000, 1000),
        "reason": "Complex analysis - need good quality"
    },
    "Step 5: Synthesis": {
        "model": "claude-sonnet-4-5-20250929",  # Final output quality
        "calls": 1,
        "tokens_per_call": (5000, 1500),
        "reason": "Final deliverable - quality important"
    }
}

optimized_total = 0
pricing_map = {
    "claude-opus-4-5-20251101": (15, 75),
    "claude-sonnet-4-5-20250929": (3, 15),
    "claude-haiku-4-5-20251001": (1, 5),
    "llama3.1:8b": (0, 0)
}

for step, config in optimized_workflow.items():
    model = config["model"]
    calls = config["calls"]
    input_tokens, output_tokens = config["tokens_per_call"]
    
    if model in pricing_map:
        input_price, output_price = pricing_map[model]
        cost = (input_tokens * input_price / 1_000_000 + 
               output_tokens * output_price / 1_000_000) * calls
    else:
        cost = 0
    
    optimized_total += cost
    
    print(f"{step}")
    print(f"  Model: {model}")
    print(f"  Calls: {calls}")
    print(f"  Cost: ${cost:.4f}")
    print(f"  Reason: {config['reason']}")
    print()

print(f"TOTAL OPTIMIZED: ${optimized_total:.4f} per workflow run")
print()

# Calculate savings
savings = expensive_total - optimized_total
savings_percent = (savings / expensive_total * 100) if expensive_total > 0 else 0

print("=" * 60)
print("SAVINGS ANALYSIS")
print("=" * 60)
print(f"Expensive workflow: ${expensive_total:.4f}")
print(f"Optimized workflow: ${optimized_total:.4f}")
print(f"Savings per run: ${savings:.4f} ({savings_percent:.1f}%)")
print()
print(f"If you run this 100x/month:")
print(f"  Expensive: ${expensive_total * 100:.2f}/month")
print(f"  Optimized: ${optimized_total * 100:.2f}/month")
print(f"  Savings: ${savings * 100:.2f}/month")
print()

# ========================================================================
# TODO: Design YOUR optimized workflow
# ========================================================================

print()
print("=" * 60)
print("YOUR TURN: Design Your Optimized Workflow")
print("=" * 60)
print()

your_workflow = """
# Research / Analytics Agent Workflow

Step 1: Data Ingestion & Preprocessing
  Model: llama3.2
  Why: Low-cost/local model is sufficient to clean, structure, and extract key fields from raw data or dashboard exports
  Estimated calls: 1
  
Step 2: Metric Extraction & Basic Summary
  Model: claude-haiku-4-5
  Why: Fast and cost-efficient for summarizing key KPIs (Throughput, Cost, OT %, Fill Rate)
  Estimated calls: 1

Step 3: Anomaly Detection (Trend & Variance Analysis)
  Model: claude-haiku-4-5
  Why: Good balance of speed and quality to detect spikes, drops, and unusual patterns
  Estimated calls: 1

Step 4: Root Cause Analysis
  Model: claude-sonnet-4-5
  Why: Requires deeper reasoning across multiple metrics and business context
  Estimated calls: 1

Step 5: Insight Prioritization
  Model: claude-sonnet-4-5
  Why: Needed to rank issues by business impact and filter noise
  Estimated calls: 1

Step 6: Executive Storytelling (WBR/MBR/QBR Output)
  Model: claude-sonnet-4-5
  Why: Produces clear, structured, stakeholder-ready narrative with actions
  Estimated calls: 1

Step 7: Stakeholder Q&A Preparation
  Model: claude-sonnet-4-5
  Why: Anticipates leadership questions and prepares strong answers
  Estimated calls: 1

Step 8: Optional Optimization Loop (Refinement)
  Model: claude-haiku-4-5 → claude-sonnet-4-5 (if needed)
  Why: Iterate cheaply first, escalate only if higher quality is required
  Estimated calls: 1–2

TOTAL ESTIMATED COST: ~$0.03–$0.06 / run
"""

print(your_workflow)

# ========================================================================
# TODO: Reflection
# ========================================================================

print()
print("=" * 60)
print("REFLECTION")
print("=" * 60)
print()

reflection = """
### Workflow Design

**Number of steps in your workflow:** 8

**Total LLM calls per workflow run:** 7–9 (depending on optional refinement step)

### Model Selection Per Step

**Step 1:** llama3.2 - Low-cost/local preprocessing is sufficient for structuring raw data  
**Step 2:** claude-haiku-4-5 - Fast and efficient for KPI summarization  
**Step 3:** claude-haiku-4-5 - good for detecting anomalies at low cost  
**Step 4:** claude-sonnet-4-5 - Needed for deeper reasoning and root cause analysis  
**Step 5:** claude-sonnet-4-5 - Prioritization requires strong business judgment  
**Step 6:** claude-sonnet-4-5 - Executive storytelling requires high-quality output  
**Step 7:** claude-sonnet-4-5 - Anticipating stakeholder questions needs strong reasoning  
**Step 8 (optional):** haiku → sonnet - Cheap iteration first, then refine if needed  

### Cost Calculation

**Total cost per workflow run:** ~$0.04

**Cost for 10 runs (development):** ~$0.40  
**Cost for 100 runs (light production):** ~$4  
**Cost for 1000 runs (heavy production):** ~$40  

### Quality vs Cost Tradeoffs

**Where did you choose cheaper models?**
- Step 1 (Preprocessing)
- Step 2 (Summarization)
- Step 3 (Anomaly Detection)

These steps are repetitive and structured, so high-end reasoning is not required.

**Where did you splurge on better models?**
- Step 4 (Root Cause Analysis)
- Step 5 (Prioritization)
- Step 6 (Executive Storytelling)
- Step 7 (Stakeholder Q&A)

These steps directly impact business decisions and stakeholder communication, so higher quality is critical.

**What's at risk if cheaper models fail?**
- Missed anomalies → important issues not detected  
- Incorrect summaries → misleading analysis downstream  
- However, risk is partially mitigated by later validation steps with stronger models  

### Optimization Opportunities

**Caching:** 
Yes — cache results for repeated datasets (e.g., same weekly reports) to avoid reprocessing

**Batch processing:** 
Yes — batch multiple metrics or time periods into a single call to reduce API usage

**Fallback strategy:** 
- If Haiku output is unclear → automatically escalate to Sonnet  
- If Sonnet fails or is too costly → retry with refined prompt or partial data  

### Budget Reality Check

**Is this affordable for your use case?** Yes

Even at 1000 runs/month (~$40), the cost is low compared to the business value of improved decision-making.

**If Yes, any room for quality upgrades?**
Yes — could:
- Use Sonnet earlier for anomaly detection in critical scenarios  
- Add a final “quality review” step using Sonnet for high-visibility reports  

### Production Readiness

**Monitoring plan:**
- Track cost per run and total monthly cost  
- Log model outputs and compare against actual data  
- Collect stakeholder feedback on insight quality  

**Alert thresholds:**
- Cost exceeds budget by >20%  
- Error rate or poor insight quality detected  
- Increased need for manual corrections  

### Biggest Insight

A hybrid model strategy delivers the best balance — most tasks do not require expensive models, but the final 20–30% of high-impact steps benefit significantly from higher-quality reasoning.
"""

print(reflection)

append_to_reflection(
    notebook="06",
    section_title="Task 3 - Budget Optimization",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")
print()

TASK 3: Optimize for Budget

WORKFLOW DESIGN

Example Research Agent Workflow:

Step 1: Search for papers (5 queries)
Step 2: Extract metadata from 20 papers
Step 3: Filter to top 5 relevant papers
Step 4: Deep analysis of each paper (5 papers)
Step 5: Synthesize findings into summary
Step 6: Generate research questions

Total steps: 6
Total LLM calls: ~31 calls


COMPARISON: Expensive vs Optimized

❌ VERSION 1: All Premium Model
------------------------------------------------------------
Step 1: Search queries
  Model: claude-opus-4-5-20251101
  Calls: 5
  Cost: $0.0525

Step 2: Metadata extraction
  Model: claude-opus-4-5-20251101
  Calls: 20
  Cost: $0.4500

Step 3: Relevance filtering
  Model: claude-opus-4-5-20251101
  Calls: 5
  Cost: $0.0975

Step 4: Deep analysis
  Model: claude-opus-4-5-20251101
  Calls: 5
  Cost: $0.6000

Step 5: Synthesis
  Model: claude-opus-4-5-20251101
  Calls: 1
  Cost: $0.1875

TOTAL EXPENSIVE: $1.3875 per workflow run


✅ VERSION 2: Optimized Mix
----

---
## 📊 Best Practices Summary

In [10]:
# Cell 9: Model Selection Best Practices
print("=" * 60)
print("MODEL SELECTION BEST PRACTICES")
print("=" * 60)
print()

best_practices = {
    "Choosing Models": [
        "✓ Match model to task complexity",
        "✓ Use cheaper models for high-volume simple tasks",
        "✓ Reserve premium models for critical decisions",
        "✓ Test on your actual data before deciding",
        "✓ Consider local models for experimentation",
        "✗ Don't use Opus for everything",
        "✗ Don't use Haiku for complex reasoning"
    ],
    
    "Cost Optimization": [
        "✓ Use local models during development",
        "✓ Cache responses when possible",
        "✓ Batch similar requests",
        "✓ Use structured outputs to reduce tokens",
        "✓ Set appropriate max_tokens limits",
        "✗ Don't over-generate (wasting output tokens)",
        "✗ Don't forget to track costs"
    ],
    
    "Quality Assurance": [
        "✓ A/B test models on same tasks",
        "✓ Measure accuracy on your specific use case",
        "✓ Have fallback to better model if needed",
        "✓ Monitor quality over time",
        "✓ Get human evaluation for critical outputs",
        "✗ Don't assume expensive = better for your task",
        "✗ Don't skip quality testing"
    ],
    
    "Production Strategy": [
        "✓ Start with balanced model (Sonnet)",
        "✓ Monitor costs and quality metrics",
        "✓ Be ready to swap models if needed",
        "✓ Document your model selection rationale",
        "✓ Review and optimize quarterly",
        "✗ Don't lock into one model without testing",
        "✗ Don't ignore cost trends"
    ]
}

for category, practices in best_practices.items():
    print(f"🎯 {category}")
    print("-" * 60)
    for practice in practices:
        print(f"  {practice}")
    print()

print()
print("=" * 60)
print("GOLDEN RULES")
print("=" * 60)
print()
print("1. Test before committing - what works in theory may not in practice")
print("2. Simple tasks don't need expensive models")
print("3. Monitor costs - they can surprise you")
print("4. Quality matters more than cost for critical tasks")
print("5. Local models are your friend during development")
print()

MODEL SELECTION BEST PRACTICES

🎯 Choosing Models
------------------------------------------------------------
  ✓ Match model to task complexity
  ✓ Use cheaper models for high-volume simple tasks
  ✓ Reserve premium models for critical decisions
  ✓ Test on your actual data before deciding
  ✓ Consider local models for experimentation
  ✗ Don't use Opus for everything
  ✗ Don't use Haiku for complex reasoning

🎯 Cost Optimization
------------------------------------------------------------
  ✓ Use local models during development
  ✓ Cache responses when possible
  ✓ Batch similar requests
  ✓ Use structured outputs to reduce tokens
  ✓ Set appropriate max_tokens limits
  ✗ Don't over-generate (wasting output tokens)
  ✗ Don't forget to track costs

🎯 Quality Assurance
------------------------------------------------------------
  ✓ A/B test models on same tasks
  ✓ Measure accuracy on your specific use case
  ✓ Have fallback to better model if needed
  ✓ Monitor quality over time
  ✓

---
## ✅ Notebook 06 Complete!

### Summary

You've mastered model comparison and selection! You now know:
- ✅ Different model tiers and their tradeoffs
- ✅ How to compare models systematically
- ✅ Cost-benefit analysis frameworks
- ✅ How to optimize workflows for budget
- ✅ When to use which model
- ✅ Production deployment strategies

In [11]:
# Final Reflection
print("=" * 60)
print("OVERALL NOTEBOOK REFLECTION")
print("=" * 60)
print()

# ============================================================================
# TODO: Final reflection on model comparison
# ============================================================================

reflection = """
### 1. Which model will you use most often?

claude-haiku-4-5 — because most of my tasks (summarization, anomaly detection, metric extraction) are high-volume and don’t require deep reasoning, making it the most cost-efficient choice.

### 2. Biggest surprise about model differences?

The biggest surprise was how much better higher-end models (claude-sonnet-4-5) are at business storytelling and prioritization—not just slightly better, but significantly more useful for stakeholder communication.

### 3. How will you balance cost vs quality?

I will use a hybrid approach:
- Start with low-cost models for most tasks
- Escalate to higher-quality models only for high-impact outputs (executive summaries, decisions)
This ensures efficiency without compromising critical insights.

### 4. Confidence in model selection? (1-5)

**Confidence:** 4/5

Confidence would increase with more real-world testing, especially validating insight accuracy and stakeholder feedback over time.

### 5. Your model selection strategy in one sentence:

Use cheap models for scale and expensive models for impact.

### 6. How will this impact your project budget?

Estimated monthly cost:
- Low usage: ~$10
- Moderate usage: ~$20–30
- High usage (1000+ runs): ~$40–50

This is affordable and scalable, especially with the hybrid routing strategy.

### 7. Key takeaway from this notebook?

Not all tasks need the best model—strategic model selection can significantly reduce cost while still delivering high-quality, business-ready insights.
"""

print(reflection)

# Save reflection
append_to_reflection(
    notebook="06",
    section_title="Overall Reflection",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")

# Show costs
print()
print("=" * 60)
print("YOUR COSTS THIS NOTEBOOK")
print("=" * 60)
print()
tracker.report()

print()
print("=" * 60)
print("✅ NOTEBOOK 06 COMPLETE!")
print("=" * 60)
print()
print("Progress: [██████████████████░░] 75% Complete")
print()
print("✓ Notebook 00: Setup Verification")
print("✓ Notebook 01: Environment Setup")
print("✓ Notebook 02: LLM Basics")
print("✓ Notebook 03: CO-STAR Framework")
print("✓ Notebook 04: Structured Outputs")
print("✓ Notebook 05: Chain of Thought")
print("✓ Notebook 06: Model Comparison ← YOU ARE HERE")
print("○ Notebook 07: MCP Introduction")
print("○ Notebook 08: Project Kickoff")
print()
print("Next: notebooks/07_mcp_introduction.ipynb")
print()

OVERALL NOTEBOOK REFLECTION


### 1. Which model will you use most often?

claude-haiku-4-5 — because most of my tasks (summarization, anomaly detection, metric extraction) are high-volume and don’t require deep reasoning, making it the most cost-efficient choice.

### 2. Biggest surprise about model differences?

The biggest surprise was how much better higher-end models (claude-sonnet-4-5) are at business storytelling and prioritization—not just slightly better, but significantly more useful for stakeholder communication.

### 3. How will you balance cost vs quality?

I will use a hybrid approach:
- Start with low-cost models for most tasks
- Escalate to higher-quality models only for high-impact outputs (executive summaries, decisions)
This ensures efficiency without compromising critical insights.

### 4. Confidence in model selection? (1-5)

**Confidence:** 4/5

Confidence would increase with more real-world testing, especially validating insight accuracy and stakeholder feedback 